In [6]:
import pandas as pd
 
# 1. Load data
df = pd.read_csv("Dataset_clean.csv", parse_dates=["order_date"])


In [7]:
# 3. Calculate R, F, M per customer
rfm = df.groupby(["customer_id", "customer_name"]).agg(
    recency=("order_date", lambda x: (snapshot_date - x.max()).days),  # days since last order
    frequency=("order_id", "nunique"),                                 # number of distinct orders
    monetary=("sales", "sum"),                                         # total spend
).reset_index()
 

In [8]:
# 4. Score each metric from 1 (worst) to 5 (best) using quintiles
#    rank(method="first") avoids errors from tied values at bin edges
rfm["r_score"] = pd.qcut(rfm["recency"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)  # recent = high
rfm["f_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["m_score"] = pd.qcut(rfm["monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
 
rfm["rfm_score"] = rfm["r_score"].astype(str) + rfm["f_score"].astype(str) + rfm["m_score"].astype(str)
rfm["rfm_total"] = rfm[["r_score", "f_score", "m_score"]].sum(axis=1)
 

In [9]:
# 5. Assign 4 segments (rules use R and the average of F and M)
def assign_segment(row):
    r, fm = row["r_score"], (row["f_score"] + row["m_score"]) / 2
    if r >= 4 and fm >= 3: return "Champions"   # recent and high value
    if r >= 3:             return "Loyal"       # still active
    if fm >= 3:            return "At Risk"     # inactive but valuable
    return "Lost"                               # inactive and low value
 
rfm["rfm_segment"] = rfm.apply(assign_segment, axis=1)
 

In [12]:
# 6. Segment summary (use this for your README and dashboard)
summary = rfm.groupby("rfm_segment").agg(
    customers=("customer_id", "count"),
    avg_recency=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean"),
    total_revenue=("monetary", "sum"),
).round(1)
summary["pct_customers"] = (summary["customers"] / summary["customers"].sum() * 100).round(1)
summary["pct_revenue"] = (summary["total_revenue"] / summary["total_revenue"].sum() * 100).round(1)
summary = summary.sort_values("total_revenue", ascending=False)
 
print(summary)

             customers  avg_recency  avg_frequency  avg_monetary  \
rfm_segment                                                        
Champions          224         26.7            8.2        4098.3   
Loyal              252         57.6            5.8        2356.0   
At Risk            132        234.1            7.5        4341.2   
Lost               185        355.7            4.0        1148.3   

             total_revenue  pct_customers  pct_revenue  
rfm_segment                                             
Champions         918020.9           28.2         40.0  
Loyal             593700.9           31.8         25.8  
At Risk           573038.8           16.6         24.9  
Lost              212440.2           23.3          9.2  


In [14]:
#export
summary.to_csv("rfm_segment_summary.csv")

In [15]:
rfm.to_csv("rfm_output.csv", index=False)